In [ ]:
# ============================================================
# WOMEN AI
# Personal AI Wellness, Lifestyle, Career and Support Assistant
# One-cell Google Colab version
# ============================================================

import json
import time
import random
import urllib.request
import urllib.error
from datetime import datetime, timedelta

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output


# ============================================================
# GLOBAL SETTINGS
# ============================================================

MODELS = [
    "gemini-3.8-flash",
    "gemini-3.5-flash-lite",
    "gemini-2.5-flash-lite"
]

API_BASE = "https://generativelanguage.googleapis.com/v1beta/models"

API_KEY = ""

user_profile = {}
chat_history = []
daily_checkin_data = {}
last_api_model = ""


# ============================================================
# BASIC HELPERS
# ============================================================

def clean(value):
    if value is None:
        return ""
    return str(value).strip()


def get_value(widget):
    return clean(widget.value)


def profile_exists():
    return bool(user_profile)


# ============================================================
# GEMINI API
# ============================================================

def call_gemini(prompt, temperature=0.7, max_output_tokens=1500):

    global last_api_model

    if not API_KEY:
        return {
            "success": False,
            "text": "Please connect Gemini first.",
            "temporary": False
        }

    last_error = ""

    for model in MODELS:

        url = API_BASE + "/" + model + ":generateContent"

        payload = {
            "contents": [
                {
                    "role": "user",
                    "parts": [
                        {
                            "text": prompt
                        }
                    ]
                }
            ],
            "generationConfig": {
                "temperature": temperature,
                "maxOutputTokens": max_output_tokens
            }
        }

        for attempt in range(4):

            request = urllib.request.Request(
                url,
                data=json.dumps(payload).encode("utf-8"),
                headers={
                    "Content-Type": "application/json",
                    "x-goog-api-key": API_KEY
                },
                method="POST"
            )

            try:

                with urllib.request.urlopen(
                    request,
                    timeout=60
                ) as response:

                    raw = response.read().decode("utf-8")
                    result = json.loads(raw)

                    candidates = result.get("candidates", [])

                    if not candidates:
                        last_error = "Gemini returned no answer."
                        break

                    content = candidates[0].get("content", {})
                    parts = content.get("parts", [])

                    answer_parts = []

                    for part in parts:

                        if "text" in part:
                            answer_parts.append(part["text"])

                    answer = "\n".join(answer_parts).strip()

                    if answer:

                        last_api_model = model

                        return {
                            "success": True,
                            "text": answer,
                            "temporary": False
                        }

                    last_error = "Gemini returned an empty response."
                    break

            except urllib.error.HTTPError as error:

                status = error.code

                try:
                    error_body = error.read().decode("utf-8")
                except Exception:
                    error_body = ""

                last_error = (
                    "HTTP Status: "
                    + str(status)
                    + "\n\n"
                    + error_body
                )

                # Temporary errors
                if status == 408 or status == 429 or status >= 500:

                    if attempt < 3:

                        wait_time = (
                            (2 ** attempt)
                            + random.uniform(0.2, 1.0)
                        )

                        time.sleep(wait_time)
                        continue

                    break

                # Permanent errors
                if status in [400, 401, 403, 404]:

                    return {
                        "success": False,
                        "text": (
                            "Gemini API Error\n\n"
                            + last_error
                        ),
                        "temporary": False
                    }

                return {
                    "success": False,
                    "text": (
                        "Gemini API Error\n\n"
                        + last_error
                    ),
                    "temporary": False
                }

            except Exception as error:

                last_error = str(error)

                if attempt < 3:

                    wait_time = (
                        (2 ** attempt)
                        + random.uniform(0.2, 1.0)
                    )

                    time.sleep(wait_time)
                    continue

                break

    return {
        "success": False,
        "text": (
            "Gemini is temporarily unavailable.\n\n"
            "The application automatically tried multiple "
            "models and retry attempts."
        ),
        "temporary": True
    }


# ============================================================
# SYSTEM PROMPT
# ============================================================

def build_system_prompt():

    profile_text = json.dumps(
        user_profile,
        indent=2,
        ensure_ascii=False
    )

    return """
You are Women AI.

You are a personalized AI assistant for women's wellness,
lifestyle, education, career development and everyday support.

USER PROFILE:
""" + profile_text + """

IMPORTANT PERSONALIZATION RULES:

- Use the profile information when relevant.
- Do not mention every profile field in every answer.
- Do not invent information.
- Do not assume the user has a medical condition.
- Ask a question only when necessary.
- Give practical beginner-friendly explanations.
- Respect the user's stated preferences.
- Keep answers relevant to the question.

WELLNESS SAFETY:

- Give general educational information.
- Do not diagnose diseases.
- Do not prescribe medication.
- Do not replace a qualified healthcare professional.
- Encourage professional care when appropriate.
- Period calculations are estimates only.
- Do not describe period estimates as contraception or
  fertility guarantees.

MENTAL WELLNESS:

- Be supportive and non-judgmental.
- Do not diagnose mental health conditions.
- If the user describes immediate danger or self-harm,
  encourage immediate emergency or professional support.

CAREER:

- Help with learning, projects, resumes, interviews,
  internships and professional communication.
- Never invent qualifications or experience.
- Never guarantee a job or income.

COMMUNICATION STYLE:

- Clear
- Friendly
- Practical
- Respectful
- Beginner-friendly
- Personalized
- No unnecessary emojis
"""


# ============================================================
# TITLE
# ============================================================

title = widgets.HTML(
    """
    <h1>WOMEN AI</h1>
    <h3>Your Personal AI Wellness and Lifestyle Assistant</h3>
    <p>
    Skin | Hair | Body | Food | Period | Fitness |
    Wellness | Fashion | Career | Resume | Interview
    </p>
    """
)


# ============================================================
# GEMINI CONNECTION
# ============================================================

api_key_box = widgets.Password(
    description="API Key:",
    placeholder="Paste your Gemini API key",
    layout=widgets.Layout(width="700px")
)

connect_button = widgets.Button(
    description="Connect Gemini",
    button_style="primary",
    layout=widgets.Layout(width="180px")
)

connection_status = widgets.Output()


def connect_gemini(button):

    global API_KEY

    with connection_status:

        clear_output()

        key = get_value(api_key_box)

        if not key:

            display(Markdown(
                "Please enter your Gemini API key."
            ))

            return

        API_KEY = key

        display(Markdown(
            "Testing Gemini connection..."
        ))

        result = call_gemini(
            "Reply with exactly: GEMINI CONNECTION SUCCESSFUL",
            temperature=0,
            max_output_tokens=30
        )

        if result["success"]:

            display(Markdown(
                "### Gemini Connected Successfully\n\n"
                "Women AI is ready."
            ))

        else:

            display(Markdown(
                "### Gemini Connection Failed\n\n"
                + result["text"]
            ))


connect_button.on_click(connect_gemini)


# ============================================================
# STEP 1 - PROFILE WIDGETS
# ============================================================

name_box = widgets.Text(
    description="Name:"
)

age_box = widgets.IntText(
    description="Age:",
    value=19
)

language_box = widgets.Dropdown(
    description="Language:",
    options=[
        "English",
        "Telugu",
        "Hindi",
        "English + Telugu",
        "English + Hindi"
    ],
    value="English"
)


# Skin

skin_type = widgets.Dropdown(
    description="Skin Type:",
    options=[
        "Oily",
        "Dry",
        "Combination",
        "Normal",
        "Sensitive",
        "Not sure"
    ]
)

skin_concerns = widgets.Textarea(
    description="Concerns:",
    layout=widgets.Layout(
        width="700px",
        height="70px"
    )
)


# Hair

hair_type = widgets.Dropdown(
    description="Hair Type:",
    options=[
        "Straight",
        "Wavy",
        "Curly",
        "Coily",
        "Not sure"
    ]
)

scalp_type = widgets.Dropdown(
    description="Scalp:",
    options=[
        "Oily",
        "Dry",
        "Normal",
        "Sensitive",
        "Dandruff-prone",
        "Not sure"
    ]
)

hair_concerns = widgets.Textarea(
    description="Concerns:",
    layout=widgets.Layout(
        width="700px",
        height="70px"
    )
)


# Body

body_skin_type = widgets.Dropdown(
    description="Body Skin:",
    options=[
        "Oily",
        "Dry",
        "Combination",
        "Normal",
        "Sensitive",
        "Not sure"
    ]
)

body_concerns = widgets.Textarea(
    description="Concerns:",
    layout=widgets.Layout(
        width="700px",
        height="70px"
    )
)


# Food

diet = widgets.Dropdown(
    description="Diet:",
    options=[
        "Vegetarian",
        "Non-vegetarian",
        "Eggetarian",
        "Vegan",
        "Mixed"
    ]
)

nutrition_goal = widgets.Text(
    description="Goal:"
)


# Period

track_period = widgets.Dropdown(
    description="Track:",
    options=[
        "Yes",
        "No",
        "Prefer not to say"
    ]
)

last_period = widgets.Text(
    description="Last Period:",
    placeholder="YYYY-MM-DD"
)

cycle_length = widgets.IntText(
    description="Cycle:",
    value=28
)

period_days = widgets.IntText(
    description="Period Days:",
    value=5
)

period_button = widgets.Button(
    description="ESTIMATE NEXT PERIOD"
)

period_output = widgets.Output()


# Fitness

fitness_level = widgets.Dropdown(
    description="Level:",
    options=[
        "Beginner",
        "Intermediate",
        "Advanced",
        "Not exercising"
    ]
)

fitness_goal = widgets.Text(
    description="Goal:"
)


# Mental wellness

stress_level = widgets.Dropdown(
    description="Stress:",
    options=[
        "Low",
        "Moderate",
        "High",
        "Very high",
        "Prefer not to say"
    ]
)

wellness_goal = widgets.Text(
    description="Goal:"
)


# Fashion

body_shape = widgets.Dropdown(
    description="Body Shape:",
    options=[
        "Hourglass",
        "Pear",
        "Rectangle",
        "Apple",
        "Inverted triangle",
        "Not sure"
    ]
)

fashion_style = widgets.Text(
    description="Style:"
)

fashion_occasion = widgets.Text(
    description="Occasion:"
)


# Career

education = widgets.Text(
    description="Education:"
)

career_interests = widgets.Text(
    description="Interests:"
)

skills = widgets.Textarea(
    description="Skills:",
    layout=widgets.Layout(
        width="700px",
        height="70px"
    )
)

career_goal = widgets.Text(
    description="Career Goal:"
)


# ============================================================
# CREATE PROFILE
# ============================================================

profile_button = widgets.Button(
    description="CREATE MY WOMEN AI PROFILE",
    button_style="success",
    layout=widgets.Layout(width="300px")
)

profile_output = widgets.Output()


def create_profile(button):

    global user_profile

    if not API_KEY:

        with profile_output:

            clear_output()

            display(Markdown(
                "Please connect Gemini first."
            ))

        return

    user_profile = {

        "personal": {
            "name": get_value(name_box),
            "age": age_box.value,
            "language": get_value(language_box)
        },

        "skin": {
            "type": get_value(skin_type),
            "concerns": get_value(skin_concerns)
        },

        "hair": {
            "type": get_value(hair_type),
            "scalp": get_value(scalp_type),
            "concerns": get_value(hair_concerns)
        },

        "body": {
            "skin_type": get_value(body_skin_type),
            "concerns": get_value(body_concerns)
        },

        "food": {
            "diet": get_value(diet),
            "goal": get_value(nutrition_goal)
        },

        "period": {
            "track": get_value(track_period),
            "last_period": get_value(last_period),
            "cycle_length": cycle_length.value,
            "period_days": period_days.value
        },

        "fitness": {
            "level": get_value(fitness_level),
            "goal": get_value(fitness_goal)
        },

        "mental_wellness": {
            "stress": get_value(stress_level),
            "goal": get_value(wellness_goal)
        },

        "fashion": {
            "body_shape": get_value(body_shape),
            "style": get_value(fashion_style),
            "occasion": get_value(fashion_occasion)
        },

        "career": {
            "education": get_value(education),
            "interests": get_value(career_interests),
            "skills": get_value(skills),
            "career_goal": get_value(career_goal)
        }
    }

    with profile_output:

        clear_output()

        display(Markdown(
            "### Profile Created Successfully"
        ))

        if get_value(name_box):

            display(Markdown(
                "Welcome **"
                + get_value(name_box)
                + "**"
            ))

        display(Markdown(
            "Women AI can now personalize your answers."
        ))


profile_button.on_click(create_profile)


# ============================================================
# PERIOD ESTIMATE
# ============================================================

def estimate_period(button):

    with period_output:

        clear_output()

        if get_value(track_period) != "Yes":

            display(Markdown(
                "Period tracking is not enabled."
            ))

            return

        try:

            last_date = datetime.strptime(
                get_value(last_period),
                "%Y-%m-%d"
            )

            days = int(cycle_length.value)

            if days < 15 or days > 60:
                raise ValueError()

            estimated_date = (
                last_date
                + timedelta(days=days)
            )

            display(Markdown(
                "### Estimated Next Period\n\n"
                "Estimated date: **"
                + estimated_date.strftime("%Y-%m-%d")
                + "**\n\n"
                "This is only an estimate. Menstrual cycles "
                "can naturally vary. This estimate must not "
                "be used as contraception or a fertility guarantee."
            ))

        except Exception:

            display(Markdown(
                "Enter the last period as YYYY-MM-DD and "
                "use a cycle length between 15 and 60 days."
            ))


period_button.on_click(estimate_period)


# ============================================================
# STEP 2 - DAILY CHECK-IN
# ============================================================

mood_box = widgets.Dropdown(
    description="Mood:",
    options=[
        "Happy",
        "Okay",
        "Low",
        "Anxious",
        "Angry",
        "Tired"
    ]
)

energy_box = widgets.Dropdown(
    description="Energy:",
    options=[
        "Very low",
        "Low",
        "Medium",
        "High",
        "Very high"
    ]
)

daily_stress_box = widgets.Dropdown(
    description="Stress:",
    options=[
        "Low",
        "Moderate",
        "High",
        "Very high"
    ]
)

sleep_box = widgets.Dropdown(
    description="Sleep:",
    options=[
        "Poor",
        "Okay",
        "Good",
        "Excellent"
    ]
)

daily_goal_box = widgets.Text(
    description="Goal:"
)

daily_notes_box = widgets.Textarea(
    description="Feeling:",
    layout=widgets.Layout(
        width="700px",
        height="80px"
    )
)

checkin_button = widgets.Button(
    description="CREATE DAILY PLAN",
    button_style="success"
)

checkin_output = widgets.Output()


# Local fallback for Daily Check-in

def create_local_daily_plan():

    name = get_value(name_box) or "there"

    mood = get_value(mood_box)
    energy = get_value(energy_box)
    stress = get_value(daily_stress_box)
    sleep = get_value(sleep_box)
    goal = get_value(daily_goal_box)

    plan = []

    plan.append(
        "### Today's Personalized Plan"
    )

    plan.append(
        "Hello **"
        + name
        + "**. Based on today's check-in, keep your plan "
        "realistic and avoid trying to complete everything at once."
    )

    if energy in ["Very low", "Low"]:

        plan.append(
            "### 1. Energy\n\n"
            "Keep today's activities light. Choose one important "
            "task and take short breaks between activities."
        )

    else:

        plan.append(
            "### 1. Energy\n\n"
            "Use your available energy for one important task "
            "before moving to smaller activities."
        )

    if goal:

        plan.append(
            "### 2. Today's Goal\n\n"
            "**"
            + goal
            + "**\n\n"
            "Break this into one small action that can realistically "
            "be completed today."
        )

    else:

        plan.append(
            "### 2. Today's Goal\n\n"
            "Choose one important academic, career or personal task."
        )

    if sleep in ["Poor", "Okay"]:

        plan.append(
            "### 3. Sleep Recovery\n\n"
            "Avoid pushing yourself late into the night. Try to "
            "keep the evening calm and aim for a consistent bedtime."
        )

    else:

        plan.append(
            "### 3. Sleep\n\n"
            "Maintain your normal sleep schedule and avoid unnecessary "
            "late-night screen time."
        )

    plan.append(
        "### 4. Food and Hydration\n\n"
        "Have regular meals, include protein-rich foods and "
        "vegetables where possible, and drink water regularly."
    )

    if energy in ["Very low", "Low"]:

        plan.append(
            "### 5. Movement\n\n"
            "Try gentle stretching or a short comfortable walk "
            "instead of an intense workout."
        )

    else:

        plan.append(
            "### 5. Movement\n\n"
            "Choose a level of physical activity that matches "
            "your normal fitness routine."
        )

    if stress in ["High", "Very high"]:

        plan.append(
            "### 6. Mental Wellness\n\n"
            "Take a few quiet minutes away from screens. "
            "Slow breathing, journaling or talking to someone "
            "you trust may help you manage the day."
        )

    else:

        plan.append(
            "### 6. Mental Wellness\n\n"
            "Take a short break during the day and give yourself "
            "time away from study or work."
        )

    plan.append(
        "### 7. Evening\n\n"
        "Review what you completed today. Do not treat unfinished "
        "tasks as failure. Choose one priority for tomorrow."
    )

    return "\n\n".join(plan)


def create_daily_checkin(button):

    global daily_checkin_data

    with checkin_output:

        clear_output()

        if not API_KEY:

            display(Markdown(
                "Please connect Gemini first."
            ))

            return

        if not profile_exists():

            display(Markdown(
                "Please create your profile first."
            ))

            return

        daily_checkin_data = {

            "date": datetime.now().strftime("%Y-%m-%d"),

            "mood": get_value(mood_box),

            "energy": get_value(energy_box),

            "stress": get_value(daily_stress_box),

            "sleep": get_value(sleep_box),

            "goal": get_value(daily_goal_box),

            "feeling": get_value(daily_notes_box)
        }

        prompt = build_system_prompt() + """

Create a personalized daily plan using ONLY the relevant
information from the user's profile and today's check-in.

TODAY'S CHECK-IN:

""" + json.dumps(
            daily_checkin_data,
            indent=2,
            ensure_ascii=False
        ) + """

IMPORTANT:

Do not automatically give skincare, haircare, fashion,
nutrition or other advice just because those fields exist
in the profile.

Only include those areas if they are relevant to today's
check-in or goal.

Do not make medical claims.

Create:

1. Main priority
2. Study or career activity if relevant
3. Food and hydration
4. Movement
5. Mental wellness
6. Evening routine

Keep it realistic for one day.
"""

        result = call_gemini(
            prompt,
            temperature=0.6,
            max_output_tokens=1600
        )

        if result["success"]:

            display(Markdown(
                result["text"]
            ))

        elif result["temporary"]:

            display(Markdown(
                "Gemini is temporarily unavailable, so Women AI "
                "created a local personalized plan instead."
            ))

            display(Markdown(
                create_local_daily_plan()
            ))

        else:

            display(Markdown(
                result["text"]
            ))


checkin_button.on_click(create_daily_checkin)


# ============================================================
# STEP 3 - PERSONALIZED AI CHAT
# ============================================================

chat_input = widgets.Textarea(
    placeholder=(
        "Ask Women AI anything about your profile, lifestyle, "
        "skin, hair, body care, food, periods, fitness, "
        "wellness, fashion, career, resume or interview preparation."
    ),
    layout=widgets.Layout(
        width="700px",
        height="100px"
    )
)

send_button = widgets.Button(
    description="ASK WOMEN AI",
    button_style="primary"
)

clear_chat_button = widgets.Button(
    description="CLEAR CHAT"
)

chat_output = widgets.Output()


def send_chat(button):

    global chat_history

    message = get_value(chat_input)

    if not message:
        return

    with chat_output:

        clear_output()

        if not API_KEY:

            display(Markdown(
                "Please connect Gemini first."
            ))

            return

        if not profile_exists():

            display(Markdown(
                "Please create your profile first."
            ))

            return

        history_text = ""

        for item in chat_history[-12:]:

            if "user" in item:

                history_text += (
                    "User: "
                    + item["user"]
                    + "\n"
                )

            if "assistant" in item:

                history_text += (
                    "Women AI: "
                    + item["assistant"]
                    + "\n"
                )

        prompt = build_system_prompt() + """

PREVIOUS CONVERSATION:

""" + history_text + """

CURRENT USER MESSAGE:

""" + message + """

Answer the current message.

Use the profile only when relevant.
Do not repeat the entire profile.
Do not ask unnecessary follow-up questions.

If the user asks about skincare, haircare, food,
fitness, periods, wellness, fashion, career,
resume or interviews, personalize the answer using
the corresponding profile information.
"""

        display(Markdown(
            "**You:** " + message
        ))

        display(Markdown(
            "**Women AI:** Thinking..."
        ))

        result = call_gemini(
            prompt,
            temperature=0.7,
            max_output_tokens=1600
        )

        clear_output()

        display(Markdown(
            "**You:** " + message
        ))

        if result["success"]:

            answer = result["text"]

            display(Markdown(
                "**Women AI:**\n\n"
                + answer
            ))

        else:

            answer = result["text"]

            display(Markdown(
                "**Women AI:**\n\n"
                + answer
            ))

        chat_history.append({
            "user": message
        })

        chat_history.append({
            "assistant": answer
        })

    chat_input.value = ""


def clear_chat(button):

    global chat_history

    chat_history = []

    with chat_output:

        clear_output()

        display(Markdown(
            "Conversation memory has been cleared."
        ))


send_button.on_click(send_chat)
clear_chat_button.on_click(clear_chat)


# ============================================================
# STEP 4 - RESUME ASSISTANT
# ============================================================

resume_input = widgets.Textarea(
    placeholder="Paste your resume here",
    layout=widgets.Layout(
        width="700px",
        height="180px"
    )
)

resume_task = widgets.Dropdown(
    description="Task:",
    options=[
        "Improve my resume",
        "Write a professional summary",
        "Improve my project descriptions",
        "Improve my skills section",
        "Make it ATS-friendly",
        "Find missing sections"
    ]
)

resume_button = widgets.Button(
    description="IMPROVE RESUME",
    button_style="primary"
)

resume_output = widgets.Output()


def analyze_resume(button):

    with resume_output:

        clear_output()

        if not API_KEY:

            display(Markdown(
                "Please connect Gemini first."
            ))

            return

        if not profile_exists():

            display(Markdown(
                "Please create your profile first."
            ))

            return

        resume = get_value(resume_input)

        if not resume:

            display(Markdown(
                "Please paste your resume first."
            ))

            return

        prompt = build_system_prompt() + """

RESUME TASK:

""" + get_value(resume_task) + """

USER'S CAREER PROFILE:

""" + json.dumps(
            user_profile.get("career", {}),
            indent=2,
            ensure_ascii=False
        ) + """

RESUME:

""" + resume + """

Improve the resume professionally.

Rules:

- Do not invent qualifications.
- Do not invent experience.
- Do not invent certifications.
- Do not invent projects.
- Keep the user's actual skills.
- Make wording professional and ATS-friendly.
"""

        display(Markdown(
            "### Resume Assistant"
        ))

        result = call_gemini(
            prompt,
            temperature=0.4,
            max_output_tokens=2000
        )

        display(Markdown(
            result["text"]
        ))


resume_button.on_click(analyze_resume)


# ============================================================
# STEP 5 - INTERVIEW ASSISTANT
# ============================================================

interview_question = widgets.Textarea(
    placeholder="Paste an interview question here",
    layout=widgets.Layout(
        width="700px",
        height="100px"
    )
)

interview_button = widgets.Button(
    description="PREPARE ANSWER",
    button_style="primary"
)

interview_output = widgets.Output()


def prepare_interview(button):

    with interview_output:

        clear_output()

        if not API_KEY:

            display(Markdown(
                "Please connect Gemini first."
            ))

            return

        if not profile_exists():

            display(Markdown(
                "Please create your profile first."
            ))

            return

        question = get_value(interview_question)

        if not question:

            display(Markdown(
                "Please enter an interview question."
            ))

            return

        prompt = build_system_prompt() + """

INTERVIEW QUESTION:

""" + question + """

USER CAREER PROFILE:

""" + json.dumps(
            user_profile.get("career", {}),
            indent=2,
            ensure_ascii=False
        ) + """

Prepare an interview answer based only on the
information available in the profile.

Give:

1. What the interviewer is asking
2. A natural answer
3. Key points to remember
4. A short answer suitable for an AI interview

Do not invent experience or qualifications.
"""

        display(Markdown(
            "### Interview Assistant"
        ))

        result = call_gemini(
            prompt,
            temperature=0.4,
            max_output_tokens=1600
        )

        display(Markdown(
            result["text"]
        ))


interview_button.on_click(prepare_interview)


# ============================================================
# STEP 6 - DASHBOARD
# ============================================================

dashboard_button = widgets.Button(
    description="SHOW MY PROFILE",
    button_style="primary"
)

dashboard_output = widgets.Output()


def generate_dashboard(button):

    with dashboard_output:

        clear_output()

        if not profile_exists():

            display(Markdown(
                "Please create your profile first."
            ))

            return

        p = user_profile

        display(Markdown(
            "### Personal Information"
        ))

        display(Markdown(
            "**Name:** "
            + str(p["personal"]["name"])
            + "\n\n"
            "**Age:** "
            + str(p["personal"]["age"])
            + "\n\n"
            "**Language:** "
            + str(p["personal"]["language"])
        ))

        display(Markdown(
            "### Skin"
        ))

        display(Markdown(
            "**Type:** "
            + p["skin"]["type"]
            + "\n\n"
            "**Concerns:** "
            + (p["skin"]["concerns"] or "Not provided")
        ))

        display(Markdown(
            "### Hair"
        ))

        display(Markdown(
            "**Type:** "
            + p["hair"]["type"]
            + "\n\n"
            "**Scalp:** "
            + p["hair"]["scalp"]
            + "\n\n"
            "**Concerns:** "
            + (p["hair"]["concerns"] or "Not provided")
        ))

        display(Markdown(
            "### Body Care"
        ))

        display(Markdown(
            "**Body Skin:** "
            + p["body"]["skin_type"]
            + "\n\n"
            "**Concerns:** "
            + (p["body"]["concerns"] or "Not provided")
        ))

        display(Markdown(
            "### Food and Nutrition"
        ))

        display(Markdown(
            "**Diet:** "
            + p["food"]["diet"]
            + "\n\n"
            "**Goal:** "
            + (p["food"]["goal"] or "Not provided")
        ))

        display(Markdown(
            "### Period and Cycle"
        ))

        display(Markdown(
            "**Tracking:** "
            + p["period"]["track"]
            + "\n\n"
            "**Cycle Length:** "
            + str(p["period"]["cycle_length"])
            + " days"
        ))

        display(Markdown(
            "### Fitness"
        ))

        display(Markdown(
            "**Level:** "
            + p["fitness"]["level"]
            + "\n\n"
            "**Goal:** "
            + (p["fitness"]["goal"] or "Not provided")
        ))

        display(Markdown(
            "### Mental Wellness"
        ))

        display(Markdown(
            "**Stress:** "
            + p["mental_wellness"]["stress"]
            + "\n\n"
            "**Goal:** "
            + (p["mental_wellness"]["goal"] or "Not provided")
        ))

        display(Markdown(
            "### Fashion"
        ))

        display(Markdown(
            "**Body Shape:** "
            + p["fashion"]["body_shape"]
            + "\n\n"
            "**Style:** "
            + (p["fashion"]["style"] or "Not provided")
            + "\n\n"
            "**Occasion:** "
            + (p["fashion"]["occasion"] or "Not provided")
        ))

        display(Markdown(
            "### Career"
        ))

        display(Markdown(
            "**Education:** "
            + (p["career"]["education"] or "Not provided")
            + "\n\n"
            "**Interests:** "
            + (p["career"]["interests"] or "Not provided")
            + "\n\n"
            "**Skills:** "
            + (p["career"]["skills"] or "Not provided")
            + "\n\n"
            "**Career Goal:** "
            + (p["career"]["career_goal"] or "Not provided")
        ))


dashboard_button.on_click(generate_dashboard)


# ============================================================
# STEP 7 - SAFETY AND SUPPORT
# ============================================================

safety_topic = widgets.Dropdown(
    description="Support:",
    options=[
        "General wellness",
        "Stress",
        "Feeling overwhelmed",
        "Mental wellness",
        "Physical health concern",
        "Emergency"
    ]
)

safety_button = widgets.Button(
    description="GET SUPPORT GUIDANCE",
    button_style="warning"
)

safety_output = widgets.Output()


def provide_safety_guidance(button):

    with safety_output:

        clear_output()

        topic = get_value(safety_topic)

        if topic == "General wellness":

            display(Markdown(
                "### General Wellness\n\n"
                "Focus on regular meals, hydration, sleep, "
                "movement and manageable daily goals."
            ))

        elif topic == "Stress":

            display(Markdown(
                "### Stress\n\n"
                "Take a short break, slow your breathing, "
                "reduce unnecessary tasks and consider speaking "
                "with someone you trust if stress continues."
            ))

        elif topic == "Feeling overwhelmed":

            display(Markdown(
                "### Feeling Overwhelmed\n\n"
                "Choose one immediate task instead of trying "
                "to solve everything at once. Break larger tasks "
                "into smaller steps and ask someone you trust "
                "for support when needed."
            ))

        elif topic == "Mental wellness":

            display(Markdown(
                "### Mental Wellness\n\n"
                "Women AI can provide general supportive information, "
                "but it cannot diagnose mental-health conditions. "
                "If difficulties persist or interfere with daily life, "
                "consider speaking with a qualified professional."
            ))

        elif topic == "Physical health concern":

            display(Markdown(
                "### Physical Health\n\n"
                "Women AI cannot diagnose or prescribe treatment. "
                "For persistent, severe or concerning symptoms, "
                "contact a qualified healthcare professional."
            ))

        elif topic == "Emergency":

            display(Markdown(
                "### Emergency Support\n\n"
                "If you are in immediate danger, seriously injured, "
                "or believe you may harm yourself or someone else, "
                "contact your local emergency service immediately "
                "and, if possible, stay with a trusted person."
            ))


safety_button.on_click(provide_safety_guidance)


# ============================================================
# DISPLAY APPLICATION
# ============================================================

display(title)


# Connection

display(
    widgets.HTML(
        "<h2>Connect Gemini AI</h2>"
        "<p>Enter your Gemini API key below.</p>"
        "<p>Do not share your API key or upload it to GitHub.</p>"
    )
)

display(
    widgets.VBox([
        api_key_box,
        connect_button,
        connection_status
    ])
)


# STEP 1

display(
    widgets.HTML(
        "<h2>STEP 1 - YOUR PROFILE</h2>"
    )
)

display(
    widgets.VBox([

        widgets.HTML(
            "<h3>Personal Information</h3>"
        ),

        name_box,
        age_box,
        language_box,

        widgets.HTML(
            "<h3>Skin Care</h3>"
        ),

        skin_type,
        skin_concerns,

        widgets.HTML(
            "<h3>Hair Care</h3>"
        ),

        hair_type,
        scalp_type,
        hair_concerns,

        widgets.HTML(
            "<h3>Body Care</h3>"
        ),

        body_skin_type,
        body_concerns,

        widgets.HTML(
            "<h3>Food and Nutrition</h3>"
        ),

        diet,
        nutrition_goal,

        widgets.HTML(
            "<h3>Period and Cycle</h3>"
        ),

        track_period,
        last_period,
        cycle_length,
        period_days,
        period_button,
        period_output,

        widgets.HTML(
            "<h3>Fitness</h3>"
        ),

        fitness_level,
        fitness_goal,

        widgets.HTML(
            "<h3>Mental Wellness</h3>"
        ),

        stress_level,
        wellness_goal,

        widgets.HTML(
            "<h3>Fashion and Styling</h3>"
        ),

        body_shape,
        fashion_style,
        fashion_occasion,

        widgets.HTML(
            "<h3>Career</h3>"
        ),

        education,
        career_interests,
        skills,
        career_goal,

        profile_button,
        profile_output
    ])
)


# STEP 2

display(
    widgets.HTML(
        "<h2>STEP 2 - DAILY WOMEN AI CHECK-IN</h2>"
    )
)

display(
    widgets.VBox([

        mood_box,
        energy_box,
        daily_stress_box,
        sleep_box,
        daily_goal_box,
        daily_notes_box,

        checkin_button,
        checkin_output
    ])
)


# STEP 3

display(
    widgets.HTML(
        "<h2>STEP 3 - PERSONALIZED AI CHAT</h2>"
    )
)

display(
    widgets.VBox([

        widgets.HTML(
            "<p>"
            "Ask Women AI about your profile, lifestyle, skin, "
            "hair, body care, food, periods, fitness, wellness, "
            "fashion, career, resume or interview preparation."
            "</p>"
        ),

        chat_input,

        widgets.HBox([
            send_button,
            clear_chat_button
        ]),

        chat_output
    ])
)


# STEP 4

display(
    widgets.HTML(
        "<h2>STEP 4 - RESUME ASSISTANT</h2>"
    )
)

display(
    widgets.VBox([
        resume_input,
        resume_task,
        resume_button,
        resume_output
    ])
)


# STEP 5

display(
    widgets.HTML(
        "<h2>STEP 5 - INTERVIEW ASSISTANT</h2>"
    )
)

display(
    widgets.VBox([
        interview_question,
        interview_button,
        interview_output
    ])
)


# STEP 6

display(
    widgets.HTML(
        "<h2>STEP 6 - MY WOMEN AI DASHBOARD</h2>"
    )
)

display(
    widgets.VBox([
        dashboard_button,
        dashboard_output
    ])
)


# STEP 7

display(
    widgets.HTML(
        "<h2>STEP 7 - SAFETY AND SUPPORT</h2>"
    )
)

display(
    widgets.VBox([
        safety_topic,
        safety_button,
        safety_output
    ])
)


# Footer

display(
    Markdown(
        """
## WOMEN AI IS READY

Profile | Skin | Hair | Body | Food | Period | Fitness | Wellness | Fashion

Career | Resume | Interview | Personalized Chat | Daily Check-in | Dashboard

**Your personalized Women AI assistant is ready.**
"""
    )
)

print("Women AI loaded successfully.")
print("Connect Gemini, create your profile, and start using the assistant.")

HTML(value='\n    <h1>WOMEN AI</h1>\n    <h3>Your Personal AI Wellness and Lifestyle Assistant</h3>\n    <p>\n…

HTML(value='<h2>Connect Gemini AI</h2><p>Enter your Gemini API key below.</p><p>Do not share your API key or u…

HTML(value='<h2>STEP 1 - YOUR PROFILE</h2>')

HTML(value='<h2>STEP 2 - DAILY WOMEN AI CHECK-IN</h2>')

HTML(value='<h2>STEP 3 - PERSONALIZED AI CHAT</h2>')

...

HTML(value='<h2>STEP 5 - INTERVIEW ASSISTANT</h2>')

HTML(value='<h2>STEP 6 - MY WOMEN AI DASHBOARD</h2>')

HTML(value='<h2>STEP 7 - SAFETY AND SUPPORT</h2>')


## WOMEN AI IS READY

Profile | Skin | Hair | Body | Food | Period | Fitness | Wellness | Fashion

Career | Resume | Interview | Personalized Chat | Daily Check-in | Dashboard

**Your personalized Women AI assistant is ready.**


Women AI loaded successfully.
Connect Gemini, create your profile, and start using the assistant.
